In [18]:
import pandas as pd
import altair as alt
from ecostyles import EcoStyles

df = pd.read_csv("../data/figure2[60].csv")
df["wealth"] = df["Real Wealth"].str.replace(".00", "", regex=False)   # £5,000.00 -> £5,000
df["year"]   = pd.to_datetime(df["yr"], format="%Y")
df["idx"]    = range(len(df))                                          # preserve mid-year step order

order   = df.sort_values("rw")["wealth"].unique().tolist()            # ascending real wealth
palette = ["#2563eb", "#4fb783", "#a8c34a", "#e6cf4a",
           "#e6a94b", "#e8843c", "#c0392b"]                            # eco blue -> red
dashes  = [[1, 0], [1, 3], [6, 3], [3, 3], [1, 2], [6, 2, 1, 2], [8, 4]]  # solid -> long-dash

styles = EcoStyles()
styles.register_and_enable_theme()

legend = alt.Legend(title="Real Wealth", orient="bottom", direction="horizontal",
                    columns=4, symbolStrokeWidth=1.8, labelLimit=120)

# hover a line -> highlight that series, dim the rest
hover = alt.selection_point(fields=["wealth"], on="pointerover", nearest=True,
                            empty=True, clear="pointerout")

enc_x = alt.X("year:T", title=None,
              axis=alt.Axis(format="%Y",
                            values=[f"{y}-01-01" for y in (1890, 1920, 1950, 1980, 2010)]))
enc_y = alt.Y("death_duty:Q", title="Death Duty, %",
              scale=alt.Scale(domain=[0, 100]), axis=alt.Axis(values=[0, 25, 50, 75, 100]))

lines = (
    alt.Chart(df)
    .mark_line(strokeWidth=1.8)
    .encode(
        x=enc_x, y=enc_y,
        color=alt.Color("wealth:N", sort=order,
                        scale=alt.Scale(domain=order, range=palette), legend=legend),
        strokeDash=alt.StrokeDash("wealth:N", sort=order,
                                  scale=alt.Scale(domain=order, range=dashes), legend=legend),
        order="idx:Q",
        opacity=alt.condition(hover, alt.value(1.0), alt.value(0.15)),
    )
)

# invisible points carry the hover + tooltip; no colour, so the legend stays clean
points = (
    alt.Chart(df)
    .mark_point(size=60)
    .encode(
        x=enc_x, y=enc_y, opacity=alt.value(0),
        tooltip=[alt.Tooltip("wealth:N", title="Real wealth"),
                 alt.Tooltip("year:T", title="Year", format="%Y"),
                 alt.Tooltip("death_duty:Q", title="Death duty, %")],
    )
    .add_params(hover)
)

chart = alt.vconcat(alt.layer(lines, points).properties(width=600, height=340))
styles.save(chart, name="uk_death_duties", svg=True)
chart

alt.VConcatChart(...)